In [3]:
from collections import Counter
import os
import cv2
import numpy as np

## Verifier le nombre des images pour chaque dossier

In [1]:
train_images = os.listdir("archive/data/train/images")
val_images = os.listdir("archive/data/val/images")
test_images = os.listdir("archive/data/test/images")

print("Train images :", len(train_images))
print("Validation images :", len(val_images))
print("Test images :", len(test_images))

Train images : 14122
Validation images : 3099
Test images : 4306


## Verifier si les images sont incorrectes

In [19]:
def verifier_images_incorrect(dataset_path):
    for i in ["train", "val", "test"]:
        image_path = os.path.join(dataset_path, i, "images")
        incorrects = []

        for img in os.listdir(image_path):
            path = os.path.join(image_path, img)
            image = cv2.imread(path)
            if image is None:
                incorrects.append(img)
                
        print(f'\nDataset: {i}')
        print(f"Images non correctes {len(incorrects)}")

verifier_images_incorrect("archive/data")


Dataset: train
Images non correctes 0

Dataset: val
Images non correctes 0

Dataset: test
Images non correctes 0


## Verifier la taille pour chaque image

In [ ]:
def verifier_taille_img(dir):
    taille = []
    for i in os.listdir(dir):
        path = os.path.join(dir, i)
        img = cv2.imread(path)

        if img is not None:
            h, w, _ = img.shape
            taille.append((w, h))

    print(f"Tailles d'images {dir} : {set(taille)}")
    print()
    print()
    
verifier_taille_img("archive/data/train/images")
verifier_taille_img("archive/data/val/images")
verifier_taille_img("archive/data/test/images")


## Verifier le format des images

In [12]:
def verifier_image_extension(dataset_path):
    image_formats = [".jpg", ".png", "jpeg"]
    for i in ["train", "val", "test"]:
        image_path = os.path.join(dataset_path, i, "images")
        out_format = []
        compte_extension = Counter()

        for img in os.listdir(image_path):
            extension = os.path.splitext(img)[1].lower()
            compte_extension[extension] += 1
            if extension not in image_formats:
                out_format.append(img)
        
            
        print(f'\nDataset: {i}')
        print(f"Image avec une autre format: {len(out_format)}")
        print(f"Images par extension: {dict(compte_extension)}")


verifier_image_extension("archive/data")


Dataset: train
Image avec une autre format: 0
Images par extension: {'.jpg': 14122}

Dataset: val
Image avec une autre format: 0
Images par extension: {'.jpg': 3099}

Dataset: test
Image avec une autre format: 0
Images par extension: {'.jpg': 4306}


## Verifier si il existe des labels qui ne porte pas le meme nom que les images (relation entre les images et les labels)

In [14]:
def verifier_images_labels(dataset_path):
    
    for i in ["train", "val", "test"]:
        image_path = os.path.join(dataset_path, i, "images")
        label_path = os.path.join(dataset_path, i, "labels")
        lst_images_without_label = []
        for img in os.listdir(image_path):
            label_file = img.replace(".jpg", ".txt")

            if not os.path.exists(os.path.join(label_path, label_file)):
                lst_images_without_label.append(img)
        
        print(f'\nDataset: {i}')
        print(f"Images sans label {len(lst_images_without_label)}")


verifier_images_labels("archive/data")


Dataset: train
Images sans label 0

Dataset: val
Images sans label 0

Dataset: test
Images sans label 0


## Verifier si il existe des labels vides

In [22]:
def verifier_labels_vide(dataset_path):
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")
        vide = []
        nb_label = []

        for file in os.listdir(label_path):
            path = os.path.join(label_path, file)
            nb_label.append(file)
            if os.path.getsize(path) == 0 : 
                vide.append(file)
                
        print(f'\nDataset: {i}')
        print(f"Fichier Label vide {len(vide)}")
        print(f'Nbr Label : {len(nb_label)}')

verifier_labels_vide("archive/data")


Dataset: train
Fichier Label vide 6458
Nbr Label : 14122

Dataset: val
Fichier Label vide 1375
Nbr Label : 3099

Dataset: test
Fichier Label vide 2005
Nbr Label : 4306


## Verifier le formats des labels (format .txt)

In [15]:
def verifier_label_format(dataset_path):
    labels_format = [".txt"]
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")
        out_format = []
        compte_extension = Counter()

        for lbl in os.listdir(label_path):
            extension = os.path.splitext(lbl)[1].lower()
            compte_extension[extension] += 1
            if extension not in labels_format:
                out_format.append(lbl)
        
            
        print(f'\nDataset: {i}')
        print(f"label avec une autre format: {len(out_format)}")
        print(f"label par extension: {dict(compte_extension)}")


verifier_label_format("archive/data")


Dataset: train
label avec une autre format: 0
label par extension: {'.txt': 14122}

Dataset: val
label avec une autre format: 0
label par extension: {'.txt': 3099}

Dataset: test
label avec une autre format: 0
label par extension: {'.txt': 4306}


## Verifier le contenu de chaque labels si il est correct

In [7]:
def verifier_label(dataset_path):
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")
        erreurs = []
        
        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            with open(file_path) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) != 5:
                        erreurs.append(file)
                        continue

                    _,x , y, w, h = map(float, parts)
                
                    if not (0<= x <= 1 and 0<= y <= 1 and 0<= w <= 1 and 0<= h <= 1):
                        erreurs.append(file)
                    
        print(f"\nDataset: {i}")
        print(f"Labels incorrects : {len(erreurs)}")


verifier_label("archive/data")


Dataset: test
Labels incorrects : 8


## Courrection des labels

In [13]:
import os

def corrige_label(dataset_path):
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")

        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            new_lines = []

            with open(file_path) as f:
                for line in f:
                    parts = line.split()

                    if len(parts) != 5:
                        continue

                    try:
                        cls = int(parts[0])
                        x, y, w, h = map(float, parts[1:])
                    except:
                        continue

                    if cls in [0, 1] and all(0 <= v <= 1 for v in [x, y, w, h]):
                        new_lines.append(line)

            with open(file_path, "w") as f:
                f.writelines(new_lines)

        print(f"Dataset {i} corrigé")

corrige_label("archive/data")

Dataset train corrigé
Dataset val corrigé
Dataset test corrigé


## On refait la verification

In [2]:
def verifier_label(dataset_path):
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")
        erreurs = []
        
        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            with open(file_path) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) != 5:
                        erreurs.append(file)
                        continue

                    _,x , y, w, h = map(float, parts)
                
                    if not (0<= x <= 1 and 0<= y <= 1 and 0<= w <= 1 and 0<= h <= 1):
                        erreurs.append(file)
                    
        print(f"\nDataset: {i}")
        print(f"Labels incorrects : {len(erreurs)}")


verifier_label("archive/data")


Dataset: train
Labels incorrects : 0

Dataset: val
Labels incorrects : 0

Dataset: test
Labels incorrects : 0


## Compte le nombre d'objet de chaque classe pour chaque dossier

In [30]:
def compte_objet_classe(dataset_path):
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")
        counter = Counter()

        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            with open(file_path) as f :
                for line in f:
                    class_id = int(line.split()[0])
                    counter[class_id] += 1
        print(f'\nDataset: {i}')
        print(f'objet par classe : {dict(counter)}')
        print(f"Smoke (0): {counter[0]}")
        print(f"Fire (1): {counter[1]}")
        

compte_objet_classe("archive/data")


Dataset: train
objet par classe : {1: 9638, 0: 7794}
Smoke (0): 7794
Fire (1): 9638

Dataset: val
objet par classe : {0: 1756, 1: 2176}
Smoke (0): 1756
Fire (1): 2176

Dataset: test
objet par classe : {0: 2315, 1: 2878}
Smoke (0): 2315
Fire (1): 2878


## Verification de bounding box (le nombre moyen pour chaque image)

In [4]:
def bounding_box_par_image(dataset_path):
    for i in ["train", "val", "test"]:
        boxes = []
        label_path = os.path.join(dataset_path, i, "labels")
        print(f'\nDataset: {i}')
        
        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            with open(file_path) as f :
                lines = f.readlines()
                boxes.append(len(lines))
                if boxes:
                    moyenne = sum(boxes) / len(boxes)
        print(f"Moyenne de nombre de bounding box par image : {moyenne:.2f}")

bounding_box_par_image("archive/data")


Dataset: train
Moyenne de nombre de bounding box par image : 1.23

Dataset: val
Moyenne de nombre de bounding box par image : 1.27

Dataset: test
Moyenne de nombre de bounding box par image : 1.20


## Calcule de surface de bounding box

In [5]:
def calcul_surface(dataset_path):
    surfaces = []
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")

        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            with open(file_path) as f:
                for line in f:
                    _, _, _, w, h = map(float, line.split())

                    surface = w * h
                    surfaces.append(surface)

    return surfaces

## Seuils pour classer les objets

In [6]:
def calcul_seuils(surfaces):
    s1 = np.percentile(surfaces, 25)
    s2 = np.percentile(surfaces, 50)
    s3 = np.percentile(surfaces, 75)

    print("Seuil petit : ", round(s1, 4))
    print("Seuil moyen : ", round(s2, 4))
    print("Seuil grand : ", round(s3, 4))

    return s1, s3

dataset_path = "archive/data"
surfaces = calcul_surface(dataset_path)
s1, s2 = calcul_seuils(surfaces)

Seuil petit :  0.003
Seuil moyen :  0.0141
Seuil grand :  0.1349


## Classification des bounding boxs

In [7]:
def taille_bounding_boxes(dataset_path):
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")

        small, medium, large = 0, 0, 0
        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            with open(file_path) as f:
                for line in f:
                    _, _, _, w, h = map(float, line.split())

                    surface = w * h

                    if surface < 0.003:
                        small += 1
                    elif surface < 0.13:
                        medium += 1
                    else:
                        large += 1

        print(f"\nDataset: {i}")
        print(f"Small: {small}, medium: {medium}, large: {large}")

taille_bounding_boxes("archive/data")


Dataset: train
Small: 4345, medium: 8676, large: 4411

Dataset: val
Small: 930, medium: 1980, large: 1022

Dataset: test
Small: 1310, medium: 2571, large: 1312


## Analyse du ratio largeur/hauteur (On regarde la forme des objets horizontaux ou verticaux)

In [9]:
def ratio_bounding_boxes(dataset_path):
    for i in ["train", "val", "test"]:
        label_path = os.path.join(dataset_path, i, "labels")

        largeur, hauteur = 0, 0
        
        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            with open(file_path) as f:
                for line in f:
                    _, _, _, w, h = map(float, line.split())

                    if w > h:
                        largeur += 1
                    else:
                        hauteur += 1

        print(f"\nDataset: {i}")
        print(f"Largeur: {largeur}, hauteur: {hauteur}")

ratio_bounding_boxes("archive/data")


Dataset: train
Largeur: 7165, hauteur: 10267

Dataset: val
Largeur: 1587, hauteur: 2345

Dataset: test
Largeur: 2163, hauteur: 3030
